In [5]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv('hepatitis.csv')

In [7]:
df.head()

,class,age,sex,steroid,antivirals,fatigue,malaise,anorexia,liver_big,liver_firm,spleen_palpable,spiders,ascites,varices,bilirubin,alk_phosphatase,sgot,albumin,protime,histology
0,1,51,1,1,2,1,2,1,2,2,1,1,2,2,?,?,?,?,?,1
1,2,30,2,1,2,2,2,2,1,2,2,2,2,2,1.00,85,18,4.0,?,1
2,2,50,1,1,2,1,2,2,1,2,2,2,2,2,0.90,135,42,3.5,?,1
3,2,78,1,2,2,1,2,2,2,2,2,2,2,2,0.70,96,32,4.0,?,1
4,2,31,1,?,1,2,2,2,2,2,2,2,2,2,0.70,46,52,4.0,80,1


In [8]:
df.shape

(155, 20)

In [9]:
df.replace('?', np.nan, inplace=True)

# Convert all applicable columns to numeric (ignore errors from bool/categorical columns)
for col in ['bilirubin', 'alk_phosphate', 'sgot', 'albumin', 'protime']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove rows with negative values in numeric columns
numeric_cols = ['bilirubin', 'alk_phosphate', 'sgot', 'albumin', 'protime']
df = df[(df[numeric_cols] >= 0).all(axis=1)]

# Drop or impute missing values (drop for simplicity)
df_cleaned = df.dropna()

KeyError: 'alk_phosphate'

In [ ]:
df_cleaned.shape

(80, 20)

In [ ]:
def remove_outliers_iqr(df, columns):
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    return df

df_no_outliers = remove_outliers_iqr(df_cleaned, ['bilirubin', 'alk_phosphate', 'sgot', 'albumin', 'protime'])

In [ ]:
df_no_outliers.shape

(54, 20)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode 'sex', 'class' and other binary/categorical columns
label_cols = ['sex', 'steroid', 'antivirals', 'fatigue', 'malaise', 'anorexia', 'liver_big', 
              'liver_firm', 'spleen_palpable', 'spiders', 'ascites', 'varices', 'histology', 'class']

for col in label_cols:
    df_no_outliers[col] = LabelEncoder().fit_transform(df_no_outliers[col].astype(str))


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

# Feature selection
X = df_no_outliers.drop('class', axis=1)
y = df_no_outliers['class']  # 0 = die, 1 = live after encoding

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# Logistic Regression
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
y_pred_logreg = logreg.predict(X_test)
logreg_acc = accuracy_score(y_test, y_pred_logreg)

# Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
nb_acc = accuracy_score(y_test, y_pred_nb)

# Compare
print("Logistic Regression Accuracy:", logreg_acc)
print("Naive Bayes Accuracy:", nb_acc)
print("\nClassification Report (LogReg):\n", classification_report(y_test, y_pred_logreg))
print("\nClassification Report (Naive Bayes):\n", classification_report(y_test, y_pred_nb))


Logistic Regression Accuracy: 0.8181818181818182
Naive Bayes Accuracy: 0.8181818181818182

Classification Report (LogReg):
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.90      0.90      0.90        10

    accuracy                           0.82        11
   macro avg       0.45      0.45      0.45        11
weighted avg       0.82      0.82      0.82        11


Classification Report (Naive Bayes):
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.90      0.90      0.90        10

    accuracy                           0.82        11
   macro avg       0.45      0.45      0.45        11
weighted avg       0.82      0.82      0.82        11

